In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '../..')
from src.models.transformers.deberta import TransformerTrainer, LLMDataset
from src.evaluation.metrics import save_results
import pandas as pd, numpy as np, pickle

CLASSES = ['gpt2','llama-chat','human','chatgpt','mistral','gpt4',
           'mpt-chat','mistral-chat','gpt3','mpt','cohere-chat','cohere']
EXPERIMENT_NAME = "deberta_base_head_only"   # Change per notebook

# ── Cell 2: Load full dataset ───────────────────────────────────────────────
df_tr = pd.read_parquet('../../data/raw/train/train.parquet')
df_vl = pd.read_parquet('../../data/raw/val/val.parquet')
df_ts = pd.read_parquet('../../data/raw/test/test.parquet')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(CLASSES)
X_tr, y_tr = df_tr['generation'].values, le.transform(df_tr['model'].values)
X_vl, y_vl = df_vl['generation'].values, le.transform(df_vl['model'].values)
X_ts, y_ts = df_ts['generation'].values, le.transform(df_ts['model'].values)

print(f"Train: {len(X_tr):,}  Val: {len(X_vl):,}  Test: {len(X_ts):,}")

# ── Cell 3: Train ───────────────────────────────────────────────────────────
trainer = TransformerTrainer(
    model_name='microsoft/deberta-v3-base',
    n_classes=12,
    strategy='head_only'   # Change: head_only | top2 | top4 | top6 | full
)
trainer.build()
trainer.train(
    X_tr, y_tr, X_vl, y_vl,
    epochs=4, batch_size=16, grad_accum=4,
    lr=2e-5, save_path=f'../../models/checkpoints/{EXPERIMENT_NAME}.pt'
)

# ── Cell 4: Evaluate ───────────────────────────────────────────────────────
test_acc, test_f1 = trainer.full_report(X_ts, y_ts, CLASSES)

# ── Cell 5: Save results ────────────────────────────────────────────────────
results = {
    'experiment': EXPERIMENT_NAME,
    'strategy': 'head_only',
    'model': 'deberta-v3-base',
    'train_acc': trainer.history[-1].get('val_acc'),  # approx — compute separately if needed
    'val_acc': max(h['val_acc'] for h in trainer.history),
    'test_acc': test_acc,
    'test_f1': test_f1,
}
import json
with open(f'../../experiments/results/{EXPERIMENT_NAME}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(results)